# 01 — Scraping immobilier : LeBonCoin + BienIci

**Objectif** : collecter des annonces immobilières de vente sur 5 villes françaises (Paris, Lyon, Bordeaux, Marseille, Nantes) depuis deux sources web, et produire deux CSV bruts prêts à être nettoyés avec PySpark (notebook 02).

**Sources retenues** :
- **LeBonCoin** — premier site de petites annonces en France, large volume d'offres immobilières
- **BienIci** — agrégateur immobilier avec des descriptions détaillées

**Outil de scraping** : [Firecrawl](https://firecrawl.dev) (SDK Python v4), qui convertit les pages web en markdown structuré exploitable par regex.

**Pipeline** :

```
Phase 1 — Découverte des URLs d'annonces
   scrape() sur les pages de résultats paginées (20 pages/ville pour LBC, 15 pour BienIci)
   → extraction des URLs via regex dans le markdown

Phase 2 — Scraping par lot des annonces
   batch_scrape() parallèle côté Firecrawl

Phase 3 — Parsing markdown → champs structurés
   Regex spécifiques par source

Phase 4 — Sauvegarde CSV
```

## 0. Imports et setup

In [ ]:
import os
import re
import time
from datetime import date
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from firecrawl import Firecrawl

# .env à la racine du repo
load_dotenv(Path('../../.env'))
API_KEY = os.getenv('FIRECRAWL_API_KEY')
assert API_KEY, "FIRECRAWL_API_KEY manquante dans .env"

app = Firecrawl(api_key=API_KEY)

# Dossiers de sortie
RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Clé API chargée : {API_KEY[:8]}...")
print(f"Dossier de sortie : {RAW_DIR.resolve()}")

## 1. Helpers génériques

Nettoyage des nombres français (espaces insécables, virgules) et retry automatique en cas de timeout réseau.

In [ ]:
def clean_num(s):
    """Convertit une string style '399 000' ou '1,5k' en float."""
    if s is None:
        return None
    cleaned = re.sub(r'[^\d.,]', '', s).replace(',', '.')
    try:
        return float(cleaned)
    except ValueError:
        return None


def scrape_with_retry(url, max_retries=2, **kwargs):
    """Scrape une URL avec retry basique en cas d'erreur ponctuelle."""
    for attempt in range(max_retries + 1):
        try:
            return app.scrape(url, formats=['markdown'], **kwargs)
        except Exception as e:
            if attempt == max_retries:
                print(f"    ✗ Échec après {max_retries+1} tentatives : {type(e).__name__}")
                return None
            time.sleep(2)

## 2. Phase 1 — Découverte des URLs d'annonces

Scrape les pages de résultats paginées et extrait les URLs d'annonces individuelles via une regex propre à chaque source. Le nombre de pages par ville est paramétrable (20 pour LeBonCoin, 15 pour BienIci).

In [ ]:
def discover_urls(search_urls, url_pattern, pages_per_city=3, source_name=""):
    """
    search_urls : dict {ville: url_template avec {page}}
    url_pattern : regex pour identifier une URL d'annonce dans le markdown
    """
    print(f"\n=== {source_name} ===")
    print("--- Phase 1 : Découverte des URLs ---\n")
    all_urls = []
    seen = set()
    pattern = re.compile(url_pattern)

    for ville, tpl in search_urls.items():
        city_count_before = len(seen)
        for p in range(1, pages_per_city + 1):
            url = tpl.format(page=p)
            result = scrape_with_retry(url)
            if result is None or not getattr(result, 'markdown', None):
                print(f"   {ville} p{p} : ÉCHEC")
                continue
            found = pattern.findall(result.markdown)
            # findall peut retourner des tuples si la regex a des groupes
            found = [f if isinstance(f, str) else f[0] for f in found]
            new = [u for u in found if u not in seen]
            for u in new:
                seen.add(u)
                all_urls.append({'url': u, 'ville_recherche': ville})
            print(f"   {ville} p{p} : {len(found)} URLs")
            time.sleep(1)  # petite politesse
        city_new = len(seen) - city_count_before
        print(f"   -> {ville:<11}: {city_new} URLs uniques\n")

    print(f"Total URLs découvertes : {len(all_urls)}")
    return all_urls

## 3. Phase 2 — Batch scraping des annonces

Utilisation de `batch_scrape` (Firecrawl) pour paralléliser les requêtes côté serveur.

In [ ]:
def batch_scrape_urls(urls, max_annonces=1000):
    """Batch scrape les URLs via Firecrawl et retourne la liste des résultats bruts."""
    urls_to_scrape = [u['url'] for u in urls[:max_annonces]]
    print(f"\n--- Phase 2 : Batch scrape de {len(urls_to_scrape)} annonces ---")
    job = app.batch_scrape(urls_to_scrape, formats=['markdown'], poll_interval=5)
    print(f"Statut : {job.status} | {job.completed}/{job.total} terminées")
    return job.data

## 4. Parsers spécifiques

### 4.1 Parser LeBonCoin

Champs extraits depuis le markdown d'une annonce LBC :
- **type_bien** / **nb_pieces** / **surface** → titre H1
- **prix** → ligne de prix sous le titre
- **ville** / **code_postal** → lien de localisation (avec fallback via l'URL « Simuler mon prêt »)
- **nb_chambres** → section caractéristiques
- **description** → section `## Description`

In [ ]:
def parse_lbc(md, url):
    """Parse un markdown Firecrawl LBC en dict structuré."""
    r = {
        'source': 'leboncoin',
        'url_annonce': url,
        'type_bien': None, 'prix': None, 'surface': None,
        'nb_pieces': None, 'nb_chambres': None,
        'ville': None, 'code_postal': None,
        'description': None,
        'date_scraping': date.today().isoformat(),
    }
    if not md:
        return r

    # --- Titre : type + pièces + surface ---
    m = re.search(
        r'#\s+(Appartement|Maison|Studio|Loft|Duplex|Parking|Terrain|Immeuble)'
        r'\s*(\d+)?\s*pi[eè]ces?\s*(\d+)\s*m²',
        md, re.IGNORECASE
    )
    if m:
        r['type_bien'] = m.group(1).lower()
        r['nb_pieces'] = int(m.group(2)) if m.group(2) else 1
        r['surface'] = float(m.group(3))
    else:
        # Cas studio qui n'a parfois pas le mot 'pièces'
        m = re.search(r'#\s+(Studio)[^\n]*?(\d+)\s*m²', md, re.I)
        if m:
            r['type_bien'] = 'studio'
            r['nb_pieces'] = 1
            r['surface'] = float(m.group(2))
        else:
            # Parking/box : pas de surface claire, on marque 'autre'
            if re.search(r'#\s+(Parking|Box|Garage)', md, re.I):
                r['type_bien'] = 'autre'

    # --- Prix : première occurrence de '999 999 €' après le H1 ---
    m = re.search(r'#\s+[^\n]+\n+([\d\s\xa0\u202f]+)\s*€', md)
    if m:
        r['prix'] = clean_num(m.group(1))

    # --- Localisation : '[Paris 75017 · Quartier ...]' ---
    m = re.search(r'\[([A-Za-zÀ-ÿ][A-Za-zÀ-ÿ\- ]+?)\s+(\d{5})', md)
    if m:
        r['ville'] = m.group(1).strip()
        r['code_postal'] = int(m.group(2))
    # Fallback : zip_code=75017&city=Paris dans l'URL 'Simuler mon prêt'
    if r['ville'] is None or r['code_postal'] is None:
        m = re.search(r'zip_code=(\d{5})&[^"\s]*city=([A-Za-z%\-]+)', md)
        if m:
            r['code_postal'] = int(m.group(1))
            r['ville'] = m.group(2).replace('%20', ' ')

    # --- Nombre de chambres ---
    m = re.search(r'Nombre de chambres\s*\n+\s*(\d+)\s*ch', md)
    if m:
        r['nb_chambres'] = int(m.group(1))

    # --- Description : ## Description → Voir plus / Détail du prix ---
    m = re.search(
        r'## Description\s*\n+(.+?)(?:\n\s*Voir plus|\n\s*Détail du prix)',
        md, re.DOTALL
    )
    if m:
        desc = m.group(1).strip()
        # Supprimer le titre répété en tête de description
        desc = re.sub(r'^(?:Appartement|Maison|Studio)[^\n]*\n+', '', desc)
        r['description'] = desc.strip()

    return r

### 4.2 Parser BienIci

Le format BienIci concentre les métadonnées dans le titre H1.

Champs extraits :
- **type_bien** / **nb_pieces** / **surface** / **code_postal** / **ville** → titre H1
- **prix** → ligne de prix
- **nb_chambres** → première occurrence dans le corps du markdown
- **description** → section `## Descriptif` jusqu'au marqueur de fin de contenu

In [ ]:
def parse_bienici(md, url):
    """Parse un markdown Firecrawl BienIci en dict structuré."""
    r = {
        'source': 'bienici',
        'url_annonce': url,
        'type_bien': None, 'prix': None, 'surface': None,
        'nb_pieces': None, 'nb_chambres': None,
        'ville': None, 'code_postal': None,
        'description': None,
        'date_scraping': date.today().isoformat(),
    }
    if not md:
        return r

    # --- Titre H1 : extrait type + pièces + surface + CP + ville ---
    m = re.search(
        r'#\s+Achat\s+(appartement|maison|studio|loft|duplex|terrain|parking|immeuble)'
        r'\s+(\d+)?\s*pi[eè]ces?\s*(\d+)\s*m²'
        r'\s*(\d{5})\s+'
        r'([A-Za-zÀ-ÿ][A-Za-zÀ-ÿ\- ]+?)'
        r'(?:\s+\d+[eèa-zéèê]+)?'   # "13e", "1er", etc.
        r'(?:\s*\(|$|\n)',
        md, re.IGNORECASE
    )
    if m:
        r['type_bien'] = m.group(1).lower()
        r['nb_pieces'] = int(m.group(2)) if m.group(2) else 1
        r['surface'] = float(m.group(3))
        r['code_postal'] = int(m.group(4))
        r['ville'] = m.group(5).strip()

    # --- Prix : 'XXX XXX €YYY €/m²' ---
    m = re.search(r'([\d\s\xa0\u202f]+)€[\s\xa0\u202f\d,k]+€/m²', md)
    if m:
        r['prix'] = clean_num(m.group(1))

    # --- Nombre de chambres ---
    m = re.search(r'\b(\d+)\s+chambres?\b', md)
    if m:
        r['nb_chambres'] = int(m.group(1))

    # --- Description : ## Descriptif... → Je souhaite être recontacté ---
    m = re.search(
        r'## Descriptif[^\n]*\n+(.+?)Je souhaite être recontacté',
        md, re.DOTALL
    )
    if m:
        r['description'] = m.group(1).strip()

    return r

## 5. Pipeline complet pour une source

In [ ]:
def collect_source(search_urls, url_pattern, parse_fn, source_name,
                   pages_per_city=10, max_annonces=5000):
    # Phase 1 : découverte
    urls = discover_urls(search_urls, url_pattern, pages_per_city, source_name)
    if not urls:
        print("Aucune URL découverte → abandon")
        return []

    # Phase 2 : batch scrape
    results = batch_scrape_urls(urls, max_annonces=max_annonces)

    # Phase 3 : parsing
    print(f"\n--- Phase 3 : Parsing de {len(results)} markdowns ---")
    records = []
    for res in results:
        md = getattr(res, 'markdown', None)
        # URL source : SDK v4 expose metadata.source_url
        meta = getattr(res, 'metadata', None)
        src_url = None
        if meta is not None:
            src_url = getattr(meta, 'source_url', None) or getattr(meta, 'url', None)
        records.append(parse_fn(md, src_url))

    print(f"{len(records)} records parsés")
    return records

## 6. Lancement LeBonCoin

In [ ]:
LBC_SEARCH_URLS = {
    "Paris":     "https://www.leboncoin.fr/recherche?category=9&locations=Paris&page={page}",
    "Lyon":      "https://www.leboncoin.fr/recherche?category=9&locations=Lyon&page={page}",
    "Bordeaux":  "https://www.leboncoin.fr/recherche?category=9&locations=Bordeaux&page={page}",
    "Marseille": "https://www.leboncoin.fr/recherche?category=9&locations=Marseille&page={page}",
    "Nantes":    "https://www.leboncoin.fr/recherche?category=9&locations=Nantes&page={page}",
}

LBC_URL_PATTERN = r'https://www\.leboncoin\.fr/ad/ventes_immobilieres/\d+'

lbc_records = collect_source(
    search_urls=LBC_SEARCH_URLS,
    url_pattern=LBC_URL_PATTERN,
    parse_fn=parse_lbc,
    source_name='LeBonCoin',
    pages_per_city=20,
    max_annonces=15000,
)

## 7. Lancement BienIci

In [8]:
BIENICI_SEARCH_URLS = {
    "Paris":     "https://www.bienici.com/recherche/achat/paris-75000?page={page}",
    "Lyon":      "https://www.bienici.com/recherche/achat/lyon-69000?page={page}",
    "Bordeaux":  "https://www.bienici.com/recherche/achat/bordeaux-33000?page={page}",
    "Marseille": "https://www.bienici.com/recherche/achat/marseille-13000?page={page}",
    "Nantes":    "https://www.bienici.com/recherche/achat/nantes-44000?page={page}",
}

BIENICI_URL_PATTERN = r'https://www\.bienici\.com/annonce/(?:achat|vente)/[^\s\)"]+'

bienici_records = collect_source(
    search_urls=BIENICI_SEARCH_URLS,
    url_pattern=BIENICI_URL_PATTERN,
    parse_fn=parse_bienici,
    source_name='BienIci',
    pages_per_city=15,
    max_annonces=15000,
)


=== BienIci ===
--- Phase 1 : Découverte des URLs ---

   Paris p1 : 26 URLs
   Paris p2 : 26 URLs
   Paris p3 : 25 URLs
   Paris p4 : 26 URLs
   Paris p5 : 26 URLs
   Paris p6 : 26 URLs
   Paris p7 : 25 URLs
   Paris p8 : 26 URLs
   Paris p9 : 25 URLs
   Paris p10 : 26 URLs
   Paris p11 : 25 URLs
   Paris p12 : 26 URLs
   Paris p13 : 26 URLs
   Paris p14 : 26 URLs
   Paris p15 : 24 URLs
   -> Paris      : 384 URLs uniques

   Lyon p1 : 25 URLs
   Lyon p2 : 26 URLs
   Lyon p3 : 26 URLs
   Lyon p4 : 25 URLs
   Lyon p5 : 26 URLs
   Lyon p6 : 26 URLs
   Lyon p7 : 25 URLs
   Lyon p8 : 26 URLs
   Lyon p9 : 26 URLs
   Lyon p10 : 26 URLs
   Lyon p11 : 26 URLs
   Lyon p12 : 23 URLs
   Lyon p13 : 23 URLs
   Lyon p14 : 26 URLs
   Lyon p15 : 26 URLs
   -> Lyon       : 378 URLs uniques

   Bordeaux p1 : 26 URLs
   Bordeaux p2 : 26 URLs
   Bordeaux p3 : 25 URLs
   Bordeaux p4 : 26 URLs
   Bordeaux p5 : 24 URLs
   Bordeaux p6 : 25 URLs
   Bordeaux p7 : 26 URLs
   Bordeaux p8 : 22 URLs
   Bordeaux p

## 8. Sauvegarde des CSV bruts

Les données sont sauvegardées telles quelles — le nettoyage (doublons, valeurs manquantes, types) sera effectué par PySpark dans le notebook 02.

In [ ]:
COLUMNS = [
    'source', 'url_annonce', 'type_bien', 'prix', 'surface',
    'nb_pieces', 'nb_chambres', 'ville', 'code_postal',
    'description', 'date_scraping',
]

df_lbc = pd.DataFrame(lbc_records, columns=COLUMNS)
df_bi  = pd.DataFrame(bienici_records, columns=COLUMNS)

df_lbc.to_csv(RAW_DIR / 'leboncoin_raw.csv', index=False)
df_bi.to_csv(RAW_DIR / 'bienici_raw.csv', index=False)

print(f"✓ LBC     : {len(df_lbc):4d} lignes → {RAW_DIR / 'leboncoin_raw.csv'}")
print(f"✓ BienIci : {len(df_bi):4d} lignes → {RAW_DIR / 'bienici_raw.csv'}")

## 9. Sanity checks et bilan

Vérifications de la qualité des données et tableau bilan. Cette cellule charge directement les CSV pour pouvoir être relancée indépendamment du scraping.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
df_lbc = pd.read_csv(RAW_DIR / 'leboncoin_raw.csv')
df_bi  = pd.read_csv(RAW_DIR / 'bienici_raw.csv')

def sanity_check(df, name):
    print(f"\n{'='*70}\n  {name} — {len(df)} lignes\n{'='*70}")
    print("\n--- Taux de remplissage (%) ---")
    print((df.notna().sum() / len(df) * 100).round(1).to_string())
    print("\n--- Répartition par ville ---")
    print(df['ville'].value_counts().to_string())
    print("\n--- Répartition type_bien ---")
    print(df['type_bien'].value_counts().to_string())
    print("\n--- Stats prix / surface / pièces ---")
    print(df[['prix','surface','nb_pieces','nb_chambres']].describe().round(0).to_string())
    print("\n--- Longueur des descriptions (caractères) ---")
    lens = df['description'].dropna().astype(str).str.len()
    if len(lens):
        print(f"  min={lens.min()}  médiane={lens.median():.0f}  moy={lens.mean():.0f}  max={lens.max()}")
    print("\n--- Prix médian /m² par ville ---")
    d = df.dropna(subset=['prix','surface']).copy()
    d['prix_m2'] = d['prix'] / d['surface']
    print(d.groupby('ville')['prix_m2'].median().round(0).to_string())
    outliers = d[(d['prix_m2'] < 1000) | (d['prix_m2'] > 30000)]
    print(f"\n--- Outliers prix/m² (< 1000 ou > 30 000) : {len(outliers)} ---")
    if len(outliers):
        print(outliers[['type_bien','prix','surface','prix_m2','ville']].to_string())
    print(f"\n--- Doublons sur url_annonce : {df['url_annonce'].duplicated().sum()} ---")

sanity_check(df_lbc, 'LeBonCoin')
sanity_check(df_bi,  'BienIci')

# --- Bilan ---
def fill_rate(df, col):
    return f"{df[col].notna().sum()/len(df)*100:.0f}%"

bilan = pd.DataFrame({
    'Source':       ['LeBonCoin', 'BienIci'],
    'Annonces':     [len(df_lbc), len(df_bi)],
    'Villes':       [df_lbc['ville'].nunique(), df_bi['ville'].nunique()],
    'Prix':         [fill_rate(df_lbc, 'prix'), fill_rate(df_bi, 'prix')],
    'Surface':      [fill_rate(df_lbc, 'surface'), fill_rate(df_bi, 'surface')],
    'Nb pièces':    [fill_rate(df_lbc, 'nb_pieces'), fill_rate(df_bi, 'nb_pieces')],
    'Nb chambres':  [fill_rate(df_lbc, 'nb_chambres'), fill_rate(df_bi, 'nb_chambres')],
    'Description':  [fill_rate(df_lbc, 'description'), fill_rate(df_bi, 'description')],
    'Long. desc médiane (car)': [
        int(df_lbc['description'].dropna().astype(str).str.len().median()) if df_lbc['description'].notna().any() else 0,
        int(df_bi['description'].dropna().astype(str).str.len().median())  if df_bi['description'].notna().any() else 0,
    ],
})

print("\n=== BILAN SCRAPING ===\n")
print(bilan.to_string(index=False))
print(f"\nTotal annonces collectées : {len(df_lbc) + len(df_bi)}")